# <b>Find Contours</b>

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

img_path = 'RGB.png'

image = cv2.imread(img_path)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blurred, 50, 150)

contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

contour_image = image_rgb.copy()
cv2.drawContours(contour_image, contours, -1, (0, 0, 0), 2)

def convert_to_bytes(image):
    """OpenCV 이미지를 JPEG 바이트 배열로 변환"""
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

original_image_widget = widgets.Image(format='jpeg')
edge_image_widget = widgets.Image(format='jpeg')
contour_image_widget = widgets.Image(format='jpeg')

display(widgets.HBox([original_image_widget, edge_image_widget, contour_image_widget]))

def update_images():
    original_image_widget.value = convert_to_bytes(image_rgb)
    edge_image_widget.value = convert_to_bytes(edges)
    contour_image_widget.value = convert_to_bytes(contour_image)

update_images()


# <b>Find Specific Color Contours</b>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

def find_contours_in_color_range(image, lower_range, upper_range):
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    color_mask = cv2.inRange(hsv_image, lower_range, upper_range)
    masked_image = cv2.bitwise_and(image, image, mask=color_mask)
    gray_mask = cv2.cvtColor(masked_image, cv2.COLOR_BGR2GRAY)
    _, binary_mask = cv2.threshold(gray_mask, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    cv2.drawContours(image, contours, -1, (0, 0, 0), 3)

    return image, masked_image

image = cv2.imread('RGB.png')

lower_blue = np.array([100, 150, 50])
upper_blue = np.array([140, 255, 255])

result_image, masked_image = find_contours_in_color_range(image, lower_blue, upper_blue)

# 이미지 출력을 위한 위젯 설정
def display_images(original, masked, result):
    clear_output(wait=True)
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # 원본 이미지
    axs[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axs[0].set_title('Original')
    axs[0].axis('off')

    # 마스크된 이미지
    axs[1].imshow(cv2.cvtColor(masked, cv2.COLOR_BGR2RGB))
    axs[1].set_title('Masked')
    axs[1].axis('off')

    # 윤곽선이 그려진 이미지
    axs[2].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    axs[2].set_title('Contours')
    axs[2].axis('off')

    plt.show()

display_images(image, masked_image, result_image)


# <b>Find White Color Contours (with Video)</b>

In [ ]:
import cv2
import ipywidgets as widgets
from IPython.display import display
import time


def find_contours_in_white(image):
    """흰색 영역의 윤곽선을 찾아 원본 이미지 위에 표시합니다."""
    result_image = image.copy()

    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, binary_image = cv2.threshold(gray_image, 200, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(result_image, contours, -1, (0, 255, 0), 2)

    return result_image


def convert_to_bytes(image):
    """OpenCV 이미지를 JPEG 바이트로 변환합니다."""
    success, buffer = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, 80])

    if not success:
        return b''

    return buffer.tobytes()


def update_video_display(cap, video_widget):
    """영상을 읽어 윤곽선을 표시합니다."""

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        result_frame = find_contours_in_white(frame)

        # 이미 출력된 위젯의 이미지만 교체
        video_widget.value = convert_to_bytes(result_frame)

        time.sleep(0.03)


# 비디오 불러오기
cap = cv2.VideoCapture('img_video/testVideo.mp4')

if not cap.isOpened():
    print("영상 파일을 열 수 없습니다.")

else:
    video_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='640px', height='360px'))

    # 위젯은 반복문 밖에서 한 번만 출력
    display(video_widget)

    try:
        update_video_display(cap, video_widget)

    except KeyboardInterrupt:
        print("영상 재생을 중지했습니다.")

    finally:
        cap.release()

# <b>Set ROI</b>

In [ ]:
from picamera2 import Picamera2
import cv2
import ipywidgets as widgets
from IPython.display import display
import time


def find_contours_in_white(image):
    """흰색 영역의 윤곽선을 찾아 표시합니다."""
    result_image = image.copy()

    gray_image = cv2.cvtColor(result_image, cv2.COLOR_BGR2GRAY)
    _, binary_image = cv2.threshold(gray_image, 200, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(result_image, contours, -1, (0, 255, 0), 2)

    return result_image


def convert_to_bytes(image):
    """OpenCV 이미지를 JPEG 바이트로 변환합니다."""
    success, buffer = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, 80])

    if not success:
        return b''

    return buffer.tobytes()


def update_video_display(picam2, video_widget):
    """카메라 영상을 처리해 위젯에 표시합니다."""

    while True:
        frame = picam2.capture_array()

        height, width = frame.shape[:2]

        # 화면 아래쪽 1/3, 가로 중앙 1/3 영역
        roi_height = height // 3
        roi_width = width // 3

        y1, y2 = 2 * roi_height, height
        x1, x2 = roi_width, width - roi_width

        roi = frame[y1:y2, x1:x2]

        # ROI 안에서 흰색 윤곽선 검출
        result_roi = find_contours_in_white(roi)

        # 전체 영상에 처리 결과 적용
        output_frame = frame.copy()
        output_frame[y1:y2, x1:x2] = result_roi

        # ROI 영역 표시
        cv2.rectangle(output_frame, (x1, y1), (x2, y2), (0, 165, 255), 2)

        # 위젯의 이미지만 변경
        video_widget.value = convert_to_bytes(output_frame)

        # 약 5FPS
        time.sleep(0.2)


picam2 = Picamera2()

camera_config = picam2.create_preview_configuration(main={"size": (320, 180), "format": "BGR888"})

picam2.configure(camera_config)
picam2.start()

video_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))

# 위젯은 한 번만 출력
display(video_widget)

try:
    update_video_display(picam2, video_widget)

except KeyboardInterrupt:
    print("카메라 실행을 중지했습니다.")

finally:
    picam2.stop()
    picam2.close()

# <b>Set ROI in detail (White Color Range & 1/5)</b>

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import time

def find_contours_in_color_range(image, lower_hsv, upper_hsv):
    """지정된 HSV 색상 범위에서 윤곽선을 찾습니다."""
    result_image = image.copy()
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    color_mask = cv2.inRange(hsv_image, lower_hsv, upper_hsv)
    contours, _ = cv2.findContours(color_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(result_image, contours, -1, (0, 255, 0), 2)

    return result_image, color_mask

def convert_to_bytes(image):
    """OpenCV 이미지를 JPEG 바이트로 변환합니다."""
    success, buffer = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, 80])

    if not success:
        return b''

    return buffer.tobytes()

def update_video_display(cap, video_widget):
    """비디오에서 흰색 영역의 윤곽선을 찾아 표시합니다."""
    lower_white = np.array([0, 0, 200], dtype=np.uint8)
    upper_white = np.array([180, 30, 255], dtype=np.uint8)

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            print("영상 재생이 종료되었습니다.")
            break

        height, width = frame.shape[:2]

        # 화면 하단 1/5 영역
        roi_height = height // 5
        x1, x2 = 0, width
        y1, y2 = 4 * roi_height, height

        roi = frame[y1:y2, x1:x2]

        # 흰색 윤곽선 검출
        result_roi, color_mask = find_contours_in_color_range(roi, lower_white, upper_white)

        # 전체 프레임에 결과 적용
        output_frame = frame.copy()
        output_frame[y1:y2, x1:x2] = result_roi

        # ROI 영역 표시
        cv2.rectangle(output_frame, (x1, y1), (x2 - 1, y2 - 1), (0, 165, 255), 2)

        # 위젯 이미지만 업데이트
        video_widget.value = convert_to_bytes(output_frame)

        time.sleep(0.05)

cap = cv2.VideoCapture('img_video/testVideo.mp4')

video_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))

if not cap.isOpened():
    print("영상 파일을 열 수 없습니다.")

else:
    display(video_widget)

    try:
        update_video_display(cap, video_widget)

    except KeyboardInterrupt:
        print("영상 재생을 중지했습니다.")

    finally:
        cap.release()